# Get BDB2026 Player-Tracking Data for WRs

For all of our 2023 rookie WRs, we want to get tracking data for their player plays from the 2023 season using BDB2026 data. 

A very important piece of what we're doing here is ***determining who the CB is that's guarding them***. 

Since we want to find separation during man coverage situations, we will want to find who the CB is that's guarding them. Currently, the BDB2026 data has tracking data for all defenders on a given player. We want to determine which CB is guarding our WR of interest.

<br><br/>
#### *Initialize Environment*

In [ ]:
import sys
from pathlib import Path

# Identify Project Root
PROJECT_ROOT = next(
    parent
    for parent in [Path.cwd(), *Path.cwd().parents]
    if (parent / "pyproject.toml").exists()
)
sys.path.insert(0, str(PROJECT_ROOT))

# Import Shared Project Paths
from utils.paths import (
    CLEAN_DATA_DIR,
    BDB_2026_BEFORE_PASS_DIR,
    BDB_2026_SUPPLEMENTARY_DIR,
    SEPARATION_DATA_DIR,
)

import polars as pl 
import pandas as pd 
import numpy as np
import plotly.express as px
import plotly.io as pio
pio.renderers.default = "png"

# Import schema_map For 2027 BDB Data
import sys
from pathlib import Path
# Identify Project Root

PROJECT_ROOT = next(

    parent

    for parent in [Path.cwd(), *Path.cwd().parents]

    if (parent / "pyproject.toml").exists()

)



# Import Shared Data Schemas

sys.path.insert(0, str(PROJECT_ROOT))

from utils.schema import schema_map

<br><br/>
## Load Data
***

In [ ]:
# =======================================
# Import Parameters That Define What 
# Data We'll Examine In This Project
# =======================================

import Data_Parameters as params

<br><br/>
#### Players Data (*BDB 2027*)

In [ ]:
# =============================
# Get Player Data For 2023 Rookie WRs
# =============================

players_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "players.csv"), 
        schema_overrides = schema_map["players"]
    )
)

# -----------------------------
# Only Keep Position of Interest
# -----------------------------

players_df = (
    players_df
    .filter(pl.col("nfl_position") == params.POSITION_OF_INTEREST)
)

# -----------------------------
# Only Keep 2023 Rookies 
# -----------------------------

# We're only keeping 2023 rookies because the 2026 BDB
# player-tracking data only includes 2023 REG season

players_df = (
    players_df
    .filter(
        pl.col("draft_year") == params.DRAFT_YEAR_OF_INTEREST
    )
)

# -----------------------------
# Only Keep Needed Variables
# -----------------------------
players_df = (
    players_df
    .select(
        "nfl_id", 
        "display_name", 
        "draft_year", 
        "nfl_position"
    )
)

# -----------------------------
# Identify Players of Interest
# -----------------------------

players_of_interest = (
    players_df
    .select("nfl_id")
    .unique()
    .to_series()
    .to_list()    
)

print("Players Data:")
print(f"    - Number of Players From 2023 Rookie Class: {len(players_of_interest)}")
players_df

<br><br/>
#### BDB2027 Route Data 

From Player-Play Data (*BDB 2027*)

In [ ]:
# =============================
# Get Route Data For Routes Our 2023 Rookies Ran
# =============================

# -----------------------------
# Load Player-Play Data (BDB 2027)
# -----------------------------

bdb27_route_df = (
    pl.read_csv(
        str(CLEAN_DATA_DIR / "player_play.csv"), 
        schema_overrides = schema_map["player_play"]
    )
    # -------------------------
    # Only Keep Route Data
    # -------------------------
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "target", 
        "route_ran", 
    )
)

# -----------------------------
# Only Keep Players of Interest
# -----------------------------

bdb27_route_df = (
    bdb27_route_df
    .filter(pl.col("nfl_id").is_in(players_of_interest))
)

print("BDB2027 Route Data:")
print(f"    - Number of Players In This Dataset: {bdb27_route_df.select(pl.col("nfl_id")).n_unique()}")
print(f"    - Number of WR Player-Plays: {bdb27_route_df.select("game_id", "play_id", "nfl_id").n_unique()}")
bdb27_route_df

<br><br/>
#### BDB2026 Route Data

From Supplemental Data (*BDB 2026*)

In [ ]:
# ==================================
# Load BDB2026 Route Data
# ==================================

bdb26_route_df = (
    pl.read_csv(
        str(BDB_2026_SUPPLEMENTARY_DIR / "supplementary_data.csv"), 
        # Change "NA" to None
        null_values = {
            "play_action": "NA", 
            "dropback_distance": "NA"
        }
    )
)

# ---------------------------------
# Only Keep Helpful Variables
# ---------------------------------

bdb26_route_df = (
    bdb26_route_df
    .select(
        # ID Variables
        "game_id", 
        "play_id", 
        # Helpful Variables
        "route_of_targeted_receiver", 
        "team_coverage_man_zone", 
    )
)

print("BDB2026 Route Data:")
print(f"    - Number of Plays In This Dataset: {bdb26_route_df.select("game_id", "play_id").n_unique()}")
bdb26_route_df

<br><br/>
#### Before Pass Data (*BDB 2026*)

In [ ]:
# ====================================
# Load Before Pass Data
# ====================================

data_dir = Path(str(BDB_2026_BEFORE_PASS_DIR))

# ------------------------------------
# Load and Combine 18 Before Pass Datasets Into 1
# ------------------------------------

before_pass_df = (
    pl.concat(
        [
            pl.scan_csv(data_dir / f"input_2023_w{i:02d}.csv")
            .with_columns(pl.lit(i).alias("week"))
            for i in range(1, 19)
        ],
        how="vertical"
    )
    .collect()
)

# -----------------------------------
# Only Keep WRs of Interest & All DBs
# -----------------------------------

before_pass_df = (
    before_pass_df
    .filter(
        (pl.col("nfl_id").is_in(players_of_interest))
        |
        (pl.col("player_position").is_in(params.DEF_POSITION_OF_INTEREST))
    )
)

# ------------------------------------
# Add Game-Play and Player-Play ID
# ------------------------------------

before_pass_df = (
    before_pass_df
    .with_columns(
        # ID For Play Within A Certain Game
        game_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String), 
        # ID For Player-Play Within A Certain Game
        player_play_id = pl.col("game_id").cast(pl.String) + "_" + pl.col("play_id").cast(pl.String) + "_" + pl.col("nfl_id").cast(pl.String)
    )
    .select(
        "game_id",
        "play_id", 
        "nfl_id", 
        "frame_id", 
        "game_play_id", 
        "player_play_id",
        "play_direction", 
        "absolute_yardline_number",
        "player_name", 
        "player_position", 
        "x", 
        "y", 
        "s", 
        "a", 
        "dir", 
        "o"
    )
)

print(f"BDB2026 Player-Tracking Data For Before-Pass Time:")
print(f"    - Number of Player-Plays: {before_pass_df.select("player_play_id").n_unique()}")
before_pass_df

<br><br/>
#### Create Joined Dataset

In [ ]:
# ===================================
# Join All Datasets Together
# ===================================

# -----------------------------------
# Join BDB2026 Route Data
# -----------------------------------

bdb26_df = (
    before_pass_df
    .join(
        bdb26_route_df,
        on = ["game_id", "play_id"], 
        how = "inner"
    )
)

# -----------------------------------
# Join BDB2027 Route Data
# -----------------------------------

bdb26_df = (
    bdb26_df
    .join(
        bdb27_route_df,
        on = ["game_id", "play_id", "nfl_id"], 
        how = "left"
    )
    # -------------------------------
    # If Player Isn't In bdb26_route_df, We'll 
    # Assume They Weren't The Target
    # -------------------------------
    .with_columns(
        target = pl.col("target").fill_null(False)
    )
)

# -----------------------------------
# Only Keep Plays With Man Coverage
# -----------------------------------

bdb26_df = (
    bdb26_df
    .filter(pl.col("team_coverage_man_zone") == params.COVERAGE_OF_INTEREST)
)

# ----------------------------------
# Only Keep Plays That Last > 1.2 Second
# ----------------------------------

play_length_df = (
    bdb26_df
    .group_by("game_play_id")
    .agg(
        play_length_secs = pl.max("frame_id") / 10
    )
)

bdb26_df = (
    bdb26_df
    .join(
        play_length_df, 
        on = "game_play_id", 
        how = "inner"
    )
)

bdb26_df = (
    bdb26_df
    .filter(
        pl.col("play_length_secs") >= params.MIN_PLAY_LENGTH_SECS
    )
)

# ----------------------------------
# Only Keep Plays with 20-100 Yards To Go To Endzone
# ----------------------------------

bdb26_df = (
    bdb26_df
    .filter(
        pl.col("absolute_yardline_number").is_between(
            lower_bound = params.ABSOLUTE_YARDLINE_BOUNDARY[0], 
            upper_bound = params.ABSOLUTE_YARDLINE_BOUNDARY[1]
        )
    )
)

# -------------------------------------
# Only Keep WRs With At Least 10 Player-Plays
# -------------------------------------

players_with_sufficient_exposure = (
    bdb26_df
    .filter(pl.col("player_position") == "WR")
    .group_by("nfl_id")
    .agg(num_player_plays = pl.col("player_play_id").n_unique())
    .filter(pl.col("num_player_plays") >= params.MIN_NUM_PLAYER_PLAYS)
    .select("nfl_id")
    .to_series()
    .to_list()
)

bdb26_df = (
    bdb26_df
    .filter(
        # WRs must have sufficient exposure
        ((pl.col("player_position") == "WR") & (pl.col("nfl_id").is_in(players_with_sufficient_exposure)))
        |
        # Keep defenders as well
        (pl.col("player_position").is_in(params.DEF_POSITION_OF_INTEREST))
    )
)

# ----------------------------------
# Only Keep Needed Variables
# ----------------------------------

bdb26_df = (
    bdb26_df
    .select(
        "game_id", 
        "play_id", 
        "nfl_id", 
        "frame_id", 
        "game_play_id", 
        "player_play_id", 
        "player_name",
        "player_position", 
        "play_direction", 
        "absolute_yardline_number", 
        "team_coverage_man_zone", 
        "target", 
        "route_of_targeted_receiver", 
        "route_ran",
        "x", 
        "y", 
        "s", 
        "dir", 
        "o"
    )
)

print("2026 BDB Data:")
print(f"    - Number of Unique WRs: {bdb26_df.filter(pl.col("player_position") == "WR").select("nfl_id").n_unique()}")
print(f"    - Number of WR Player-Plays: {bdb26_df.filter(pl.col("player_position") == "WR").select("player_play_id").n_unique()}")
bdb26_df

<br><br/>
## Pair CB To WR
***

In each route that one of our WRs of interest ran during man coverage, there should be a CB whose job is to guard that WR for the play (since it's man coverage). As of right now, our tracking data includes all CBs that were on the field during each player-play for our WRs of interest. We want to figure out which of those CBs were guarding the WR.

Assigning each WR a DB is made up of the following criteria:
* Initial lateral alignment (same value of y at snap) 
* Overall percentage of time as nearest defender
* Longest uninterrupted nearest-defender run
* Body orientation toward the WR
* Movement similar to WR across play
* How often is the defender the closest defender in second half of the play
* How well does the defender track the WR's movement in the second half of the play

According to each of the criteria above, to get a high assignment score the DB should do the following: 
* Start the play at a similar lateral position
* Be frequently the closest defender during the play
* Have a relatively long uninterrupted run as the closest defender
* Have their body orientation point towards the WR
* Their movement tracks the WR's movement
* They were frequently the closest defender in the second half of the play
* Their movement tracks the WR's movement in the second half of the play

In [ ]:
# =============================
# Specify Tracking Data
# =============================
tracking_df = bdb26_df

In [ ]:
# ============================================================
# NFL WR-DEFENDER ASSIGNMENT AND SEPARATION PIPELINE
# ============================================================
#
# OBJECTIVE
# ---------
# 1. Identify the defender most likely guarding each WR.
# 2. Calculate WR-to-assigned-defender separation at every frame.
#
# INPUT
# -----
# tracking_df: Polars DataFrame containing WR tracking data and
# defensive tracking data for the relevant game-plays.
#
# frame_id = 1 represents the snap.
# Each frame represents 0.1 seconds.
#
# OUTPUTS
# -------
# wr_defender_frames_df:
#     Every candidate WR-defender pair at every observed frame.
#
# candidate_summary_df:
#     Summary features and assignment scores for every candidate.
#
# assignments_df:
#     The highest-ranked candidate defender for each WR-play.
#
# wr_separation_df:
#     Frame-level WR-to-assigned-defender separation in yards.
#
# IMPORTANT LIMITATIONS
# ---------------------
# - Assignments are inferred, not ground-truth labels.
# - The scoring weights are heuristic starting values.
# - Verify the angle convention for dir and o before using
#   the orientation and movement features.
# - This implementation ranks each WR independently. It does
#   not yet enforce joint assignments across multiple WRs.
# ============================================================


# ==================================
# CONFIGURATION
# ==================================

# Include CBs and safeties as candidate defenders.
# Modify this list if your data uses different position labels.
DEFENDER_POSITIONS = ["CB", "FS", "SS"]

# Maximum angle between a defender's body orientation and the
# bearing toward the WR for the frame to count as "facing WR".
FACING_ANGLE_THRESHOLD = 60.0

# Distance scales for transforming distances into scores.
# Smaller distances receive higher scores.
Y_DISTANCE_SCALE = 3.0
DISTANCE_SCALE = 8.0

# Tracking frequency: 10 Hz means one frame every 0.1 seconds.
SECONDS_PER_FRAME = 0.1

# Assumed angle convention for dir and o:
# 0 degrees   = +y
# 90 degrees  = +x
# 180 degrees = -y
# 270 degrees = -x



# ==================================
# STEP 1: SEPARATE WIDE RECEIVERS AND DEFENDERS
# ==================================

# ----------------------------------
# WR ID and Tracking Data
# ----------------------------------

# player_play_id uniquely identifies a particular player's
# participation in a game-play.

wr_df = (
    tracking_df
    .filter(
        pl.col("player_position") == params.POSITION_OF_INTEREST
    )
    .select(
        "game_play_id",
        "player_play_id",
        "frame_id",
        pl.col("nfl_id").alias("wr_nfl_id"),
        pl.col("player_name").alias("wr_name"),
        "play_direction",
        pl.col("x").alias("wr_x"),
        pl.col("y").alias("wr_y"),
        pl.col("dir").alias("wr_dir"),
        pl.col("o").alias("wr_o")
    )
)

# ----------------------------------
# Defender ID and Tracking Data
# ----------------------------------

# Do not join defenders on player_play_id because that ID
# represents the WR's player-play, not the defender's identity.

defender_df = (
    tracking_df
    .filter(
        pl.col("player_position").is_in(DEFENDER_POSITIONS)
    )
    .select(
        "game_play_id",
        "frame_id",
        pl.col("nfl_id").alias("defender_nfl_id"),
        pl.col("player_name").alias("defender_name"),
        pl.col("player_position").alias("defender_position"),
        pl.col("x").alias("defender_x"),
        pl.col("y").alias("defender_y"),
        pl.col("dir").alias("defender_dir"),
        pl.col("o").alias("defender_o"),
    )
)


# ==================================
# STEP 2: PAIR EACH WR WITH EVERY DEFENDER AT EACH FRAME
# ==================================

# ----------------------------------
# Join WR Data and Defender Data
# By game_play_id and frame_id
# ----------------------------------

# This produces all observed WR-Defender combinations on the
# same play at the same frame.

wr_defender_frames_df = (
    wr_df
    .join(
        defender_df,
        on=["game_play_id", "frame_id"],
        how="inner",
    )
    .with_columns(
        # Euclidean distance between the WR and defender.
        (
            (pl.col("wr_x") - pl.col("defender_x")) ** 2
            + (pl.col("wr_y") - pl.col("defender_y")) ** 2
        )
        .sqrt()
        .alias("distance"),

        # Absolute difference in lateral position.
        #
        # Small values mean the players occupy similar y
        # coordinates across the width of the field.
        (
            pl.col("wr_y") - pl.col("defender_y")
        )
        .abs()
        .alias("y_diff"),

        # Defender position relative to the WR along the
        # offense's direction of travel.
        #
        # Positive values mean the defender is farther
        # downfield than the WR in the offense's direction.
        #
        # Negative values mean the defender is behind the WR.
        #
        # The direction adjustment is necessary because x
        # increases in one direction and decreases in the other.
        (
            (
                pl.col("defender_x") - pl.col("wr_x")
            )
            *
            pl.when(pl.col("play_direction") == "right")
            .then(1.0)
            .when(pl.col("play_direction") == "left")
            .then(-1.0)
            .otherwise(None)
        )
        .alias("ahead_distance"),
    )
)


# ==================================
# STEP 3: CALCULATE DEFENDER ORIENTATION RELATIVE TO THE WR
# ==================================

# Calculate the bearing from the defender toward the WR.

# Under the assumed convention, atan2(dx, dy) calculates an
# angle measured clockwise from the positive y-axis.

# The result is converted to degrees and wrapped to [0, 360).

wr_defender_frames_df = (
    wr_defender_frames_df
    .with_columns(
        (
            pl.arctan2(
                pl.col("wr_x") - pl.col("defender_x"),
                pl.col("wr_y") - pl.col("defender_y"),
            )
            * (180.0 / 3.141592653589793)
            + 360.0
        )
        .mod(360.0)
        .alias("bearing_defender_to_wr")
    )
)

# Calculate circular angular differences.

# Circular arithmetic is important because 359 and 1 degrees
# differ by 2 degrees, not 358 degrees.

wr_defender_frames_df = (
    wr_defender_frames_df
    .with_columns(
        # Difference between defender body orientation and the
        # bearing toward the WR, expressed in [0, 180].
        (
            (
                (
                    pl.col("defender_o")
                    - pl.col("bearing_defender_to_wr")
                    + 180.0
                )
                .mod(360.0)
                + 360.0
            )
            .mod(360.0)
            - 180.0
        )
        .abs()
        .alias("facing_angle_diff"),

        # Difference between the movement headings of the WR
        # and defender, expressed in [-180, 180].
        (
            (
                (
                    pl.col("defender_dir")
                    - pl.col("wr_dir")
                    + 180.0
                )
                .mod(360.0)
                + 360.0
            )
            .mod(360.0)
            - 180.0
        )
        .alias("movement_angle_diff"),
    )
    .with_columns(
        # Does the defender's body orientation point approximately
        # toward the WR?
        (
            pl.col("facing_angle_diff")
            <= FACING_ANGLE_THRESHOLD
        )
        .alias("facing_wr"),

        # Absolute cosine of the movement-heading difference.
        #
        # Values range from 0 to 1.
        # 1 = parallel or opposite headings.
        # 0 = perpendicular headings.
        #
        # Opposite headings can still be consistent with man
        # coverage, for example when a defender backpedals.
        (
            (
                pl.col("movement_angle_diff")
                * (3.141592653589793 / 180.0)
            )
            .cos()
            .abs()
        )
        .alias("movement_similarity"),
    )
)


# ============================================================
# STEP 4: IDENTIFY THE LATTER HALF OF EACH WR-PLAY
# ============================================================

# Use each WR's observed frame range, rather than assuming
# that all plays have the same duration.
#
# Example:
#   first frame = 1
#   last frame  = 41
#   midpoint    = 21
#
# Frames at or after the midpoint are considered the latter half.

wr_defender_frames_df = (
    wr_defender_frames_df
    .with_columns(
        pl.col("frame_id")
        .min()
        .over(["game_play_id", "player_play_id"])
        .alias("first_frame"),

        pl.col("frame_id")
        .max()
        .over(["game_play_id", "player_play_id"])
        .alias("last_frame"),
    )
    .with_columns(
        (
            pl.col("frame_id")
            >= (
                pl.col("first_frame")
                + pl.col("last_frame")
            ) / 2.0
        )
        .alias("is_late_half")
    )
)


# ============================================================
# STEP 5: IDENTIFY THE NEAREST DEFENDER AT EACH FRAME
# ============================================================

# For each WR-play and frame, identify the single candidate
# defender with the smallest Euclidean distance to the WR.

nearest_by_frame_df = (
    wr_defender_frames_df
    .sort(
        [
            "game_play_id",
            "player_play_id",
            "frame_id",
            "distance",
            "defender_nfl_id",
        ]
    )
    .group_by(
        ["game_play_id", "player_play_id", "frame_id"],
        maintain_order=True,
    )
    .first()
    .select(
        "game_play_id",
        "player_play_id",
        "frame_id",
        pl.col("defender_nfl_id").alias(
            "nearest_defender_nfl_id"
        ),
    )
)

# Flag whether each candidate is the nearest defender.
# Also flag whether that candidate is nearest in the latter half.

wr_defender_frames_df = (
    wr_defender_frames_df
    .join(
        nearest_by_frame_df,
        on=["game_play_id", "player_play_id", "frame_id"],
        how="left",
    )
    .with_columns(
        (
            pl.col("defender_nfl_id")
            == pl.col("nearest_defender_nfl_id")
        )
        .alias("is_nearest")
    )
    .with_columns(
        (
            pl.col("is_nearest")
            & pl.col("is_late_half")
        )
        .alias("is_late_nearest")
    )
)


# ============================================================
# STEP 6: CALCULATE NEAREST-DEFENDER RUN DURATIONS
# ============================================================

# A run is a sequence of consecutive observed frames during
# which a particular defender remains the nearest defender.
#
# The cumulative counter creates a new run after each frame
# in which the candidate is not nearest.

nearest_streaks_df = (
    wr_defender_frames_df
    .sort(
        [
            "game_play_id",
            "player_play_id",
            "defender_nfl_id",
            "frame_id",
        ]
    )
    .with_columns(
        (
            (~pl.col("is_nearest"))
            .cast(pl.Int64)
            .cum_sum()
            .over(
                [
                    "game_play_id",
                    "player_play_id",
                    "defender_nfl_id",
                ]
            )
        )
        .alias("nearest_run_id")
    )
    .filter(pl.col("is_nearest"))
    .group_by(
        [
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "defender_nfl_id",
            "nearest_run_id",
        ]
    )
    .agg(
        pl.len().alias("run_frames")
    )
    .group_by(
        [
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "defender_nfl_id",
        ]
    )
    .agg(
        pl.col("run_frames")
        .max()
        .alias("nearest_duration_max")
    )
    .with_columns(
        (
            pl.col("nearest_duration_max")
            * SECONDS_PER_FRAME
        )
        .alias("nearest_duration_max_seconds")
    )
)


# ============================================================
# STEP 7: SUMMARIZE EACH WR-DEFENDER CANDIDATE
# ============================================================

candidate_summary_df = (
    wr_defender_frames_df
    .group_by(
        [
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "wr_name",
            "defender_nfl_id",
            "defender_name",
            "defender_position",
        ]
    )
    .agg(
        # Number of observed frames for this candidate pair.
        pl.len().alias("n_observed_frames"),

        # ----------------------------------------------------
        # WHO IS CLOSEST OVER THE ENTIRE PLAY?
        # ----------------------------------------------------

        # Fraction of all observed frames when this defender
        # is the nearest candidate to the WR.
        pl.col("is_nearest")
        .mean()
        .alias("nearest_frame_pct"),

        # ----------------------------------------------------
        # WHO IS CLOSEST DURING THE LATTER HALF?
        # ----------------------------------------------------

        # Fraction of latter-half frames when this defender
        # is nearest to the WR.
        #
        # Filter excludes the first half from the denominator.
        pl.col("is_late_nearest")
        .filter(pl.col("is_late_half"))
        .mean()
        .alias("late_nearest_frame_pct"),

        # ----------------------------------------------------
        # WHO IS MOVING WITH THE WR DURING THE LATTER HALF?
        # ----------------------------------------------------

        # Average movement-heading similarity in the latter half.
        # The underlying feature is abs(cos(angle difference)):
        # 1 means parallel OR opposite headings; 0 means
        # perpendicular headings.
        pl.col("movement_similarity")
        .filter(pl.col("is_late_half"))
        .mean()
        .alias("late_movement_similarity"),

        # Whole-play average movement-heading similarity.
        pl.col("movement_similarity")
        .mean()
        .alias("movement_similarity"),

        # Fraction of observed frames in which the defender's
        # body orientation points within the angle threshold.
        pl.col("facing_wr")
        .cast(pl.Float64)
        .mean()
        .alias("facing_wr_pct"),

        # Average lateral alignment difference.
        pl.col("y_diff")
        .mean()
        .alias("mean_y_diff"),

        # Mean distance between WR and candidate defender.
        pl.col("distance")
        .mean()
        .alias("mean_distance"),

        # Average defender position relative to the WR.
        pl.col("ahead_distance")
        .mean()
        .alias("mean_ahead_distance"),
    )
)


# Attach the longest consecutive nearest-defender run.
candidate_summary_df = (
    candidate_summary_df
    .join(
        nearest_streaks_df,
        on=[
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "defender_nfl_id",
        ],
        how="left",
    )
    .with_columns(
        pl.col("nearest_duration_max").fill_null(0),
        pl.col("nearest_duration_max_seconds").fill_null(0),
    )
)


# ============================================================
# STEP 8: CALCULATE INITIAL ALIGNMENT FEATURES
# ============================================================

# frame_id = 1 is the snap in your data.
# This measures alignment at the snap, not pre-snap alignment.

initial_alignment_df = (
    wr_defender_frames_df
    .filter(pl.col("frame_id") == 1)
    .select(
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "defender_nfl_id",
        pl.col("y_diff").alias("initial_y_diff"),
    )
)

candidate_summary_df = (
    candidate_summary_df
    .join(
        initial_alignment_df,
        on=[
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "defender_nfl_id",
        ],
        how="left",
    )
    .with_columns(
        # Convert initial lateral distance into a score from
        # approximately 0 to 1. Smaller y differences score higher.
        (
            -pl.col("initial_y_diff") / Y_DISTANCE_SCALE
        )
        .exp()
        .alias("initial_y_score"),

        # Convert the longest nearest run into a score from
        # 0 to 1, with diminishing returns for longer runs.
        (
            1.0
            - (
                -pl.col("nearest_duration_max_seconds") / 2.0
            ).exp()
        )
        .alias("nearest_duration_score"),
    )
)


# ============================================================
# STEP 9: CALCULATE THE TIME-AWARE ASSIGNMENT SCORE
# ============================================================

# All seven component scores below are bounded approximately
# between 0 and 1, assuming valid tracking values.
#
# The latter-half proximity and movement features receive
# 50% of the total weight combined.

candidate_summary_df = (
    candidate_summary_df
    .with_columns(
        (
            0.05 * pl.col("initial_y_score")
            + 0.15 * pl.col("nearest_frame_pct")
            + 0.10 * pl.col("nearest_duration_score")
            + 0.10 * pl.col("facing_wr_pct")
            + 0.10 * pl.col("movement_similarity")
            + 0.25 * pl.col("late_nearest_frame_pct")
            + 0.25 * pl.col("late_movement_similarity")
        )
        .alias("assignment_score")
    )
    .sort(
        [
            "game_play_id",
            "player_play_id",
            "assignment_score",
        ],
        descending=[False, False, True],
    )
)


# ============================================================
# STEP 10: RANK CANDIDATES AND SELECT THE TOP DEFENDER
# ============================================================

ranked_candidates_df = (
    candidate_summary_df
    .with_columns(
        pl.col("assignment_score")
        .rank(
            method="ordinal",
            descending=True,
        )
        .over(["game_play_id", "player_play_id"])
        .alias("candidate_rank")
    )
    .sort(
        [
            "game_play_id",
            "player_play_id",
            "candidate_rank",
        ]
    )
)

assignments_df = (
    ranked_candidates_df
    .filter(pl.col("candidate_rank") == 1)
    .select(
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "wr_name",
        pl.col("defender_nfl_id").alias(
            "assigned_defender_nfl_id"
        ),
        pl.col("defender_name").alias(
            "assigned_defender_name"
        ),
        pl.col("defender_position").alias(
            "assigned_defender_position"
        ),
        "assignment_score",
        "initial_y_diff",
        "initial_y_score",
        "nearest_frame_pct",
        "late_nearest_frame_pct",
        "nearest_duration_max",
        "nearest_duration_max_seconds",
        "nearest_duration_score",
        "facing_wr_pct",
        "movement_similarity",
        "late_movement_similarity",
        "mean_distance",
    )
)


# ============================================================
# STEP 11: RETRIEVE ASSIGNED DEFENDER TRACKING DATA
# ============================================================

# For each WR-play assignment, retrieve the assigned defender's
# tracking data at every observed frame.

assigned_defender_frames_df = (
    assignments_df
    .select(
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "assigned_defender_nfl_id",
        "assigned_defender_name",
        "assigned_defender_position",
        "assignment_score",
    )
    .join(
        defender_df,
        left_on=[
            "game_play_id",
            "assigned_defender_nfl_id",
        ],
        right_on=[
            "game_play_id",
            "defender_nfl_id",
        ],
        how="inner",
    )
    .select(
        "game_play_id",
        "player_play_id",
        "wr_nfl_id",
        "assigned_defender_nfl_id",
        "assigned_defender_name",
        "assigned_defender_position",
        "assignment_score",
        "frame_id",
        "defender_x",
        "defender_y",
        "defender_dir",
        "defender_o",
    )
)


# ============================================================
# STEP 12: JOIN ASSIGNED DEFENDER TO WR TRACKING
# ============================================================

# Join the selected defender's tracking coordinates to the
# corresponding WR at the same frame.
#
# The result contains one row per observed WR-frame, assuming
# the input tracking data has unique player-frame records.

wr_separation_df = (
    wr_df
    .join(
        assigned_defender_frames_df,
        on=[
            "game_play_id",
            "player_play_id",
            "wr_nfl_id",
            "frame_id",
        ],
        how="inner",
    )
    .with_columns(
        # Euclidean distance between the WR and assigned defender.
        (
            (pl.col("wr_x") - pl.col("defender_x")) ** 2
            + (pl.col("wr_y") - pl.col("defender_y")) ** 2
        )
        .sqrt()
        .alias("wr_cb_distance")
    )
    .sort(
        [
            "game_play_id",
            "player_play_id",
            "frame_id",
        ]
    )
)


# ============================================================
# STEP 13: CHECK OUTPUTS
# ============================================================

print(f"Assignments: {assignments_df.shape}")
print(f"WR separation tracking: {wr_separation_df.shape}")


In [ ]:
# ======================================
# Review Assignments, See If It Looks Accurate
# ======================================

# --------------------------------------
# Assignments Must Have This Assignment Score
# --------------------------------------

MIN_ASSIGNMENT_SCORE = 0.80

assignments_df = (
    assignments_df
    .filter(pl.col("assignment_score") >= MIN_ASSIGNMENT_SCORE)
)

print(f"Assignments That Meet Assignment Similarity Threshold:")
print(f"    - Number of WR Player-Plays: {assignments_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {assignments_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {assignments_df.select("game_play_id").n_unique()}")
assignments_df

In [ ]:
# ======================================
# Tracking Data With Pairings
# ======================================

# --------------------------------------
# Assignments Must Have This Assignment Score
# --------------------------------------

MIN_ASSIGNMENT_SCORE = 0.80

wr_separation_df = (
    wr_separation_df
    .filter(pl.col("assignment_score") >= MIN_ASSIGNMENT_SCORE)
)

print(f"Assignments That Meet Assignment Similarity Threshold:")
print(f"    - Number of WR Player-Plays: {wr_separation_df.select("player_play_id").n_unique()}")
print(f"    - Number of WRs: {wr_separation_df.select("wr_nfl_id").n_unique()}")
print(f"    - Number of Unique Plays: {wr_separation_df.select("game_play_id").n_unique()}")
wr_separation_df

In [ ]:
# ======================================
# Play Exposure
# ======================================

(
    wr_separation_df
    .group_by("wr_nfl_id")
    .agg(num_plays = pl.col("player_play_id").n_unique())
    .sort("num_plays", descending = True)
)


<br><br/>
## Export Data
***

In [ ]:
# =================================================
# Player-Time Level WR/Defender Player-Tracking & Play Data
# =================================================

# This dataset has 1 row per-player-time for all players needed to 
# create our pairings. It has useful data on things such as
#     - route
#     - target
#     - yardline
#     - coverage
#     - play_direction

bdb26_df.write_csv(str(SEPARATION_DATA_DIR / "00_2023_Rookie_Player_Time_Level_Play_Details_Data.csv"))

# =================================================
# Player-Time Level WR/CB Separtion Data
# =================================================

# This dataset has 1 row per player-time from a player-play from 
# one of our WRs of interest. It includes player-tracking data for
# the player and their assigned defender. It also includes their
# separation (distance between WR and DB) at every player-time.

wr_separation_df.write_csv(str(SEPARATION_DATA_DIR / "01_2023_Rookie_Player_Time_Level_Separation_Data.csv"))